In [ ]:
import pandas as pd
import re

# Load data
acai = pd.read_csv('../data/ACAI_ranked_universities.csv')
csr = pd.read_csv('../data/CSRankings-1_1_26.csv')
display(acai)
display(csr)

# Fix CSRankings header shift and clean institution names
csr = csr.rename(columns={'Institution': 'csrankings_rank',
                          'Count': 'Institution',
                          'Count.1': 'Count'})
csr['Institution'] = csr['Institution'].astype(str).str.replace('►', '', regex=False).str.strip()
csr['csrankings_rank'] = pd.to_numeric(csr['csrankings_rank'], errors='coerce')
csr = csr.dropna(subset=['Institution', 'csrankings_rank'])
csr = csr[csr['Institution'].astype(str).str.strip().ne('')]

def norm(s):
    s = str(s).lower()
    s = s.replace('&', ' and ')
    s = re.sub(r'\buniv\.?\b', 'university', s)
    s = re.sub(r'[^a-z0-9 ]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

# Normalize names for matching
acai['Institution_norm'] = acai['Institution'].map(norm)
csr['Institution_norm'] = csr['Institution'].map(norm)

# Merge csrankings rank into ACAI
csr_rank = csr[['Institution_norm', 'csrankings_rank']].drop_duplicates('Institution_norm')
merged = acai.merge(csr_rank, on='Institution_norm', how='left').drop(columns=['Institution_norm'])

# Save merged output
merged.to_csv('../data/ACAI_CSRankings.csv', index=False)

# Save unmatched institutions
unmatched = merged[merged['csrankings_rank'].isna()][['Institution']]
unmatched.to_csv('../data/ACAI_CSRankings_unmatched.csv', index=False)

print('Wrote data/ACAI_CSRankings.csv')
print('Wrote data/ACAI_CSRankings_unmatched.csv')
print('Unmatched count, review manually:', unmatched.shape[0])


,Rank,Percentile_Rank,Institution,Index,Type,Research Activity,Regional Coverage,size_bucket,ACAI_indicators
0,1,1,University of New Hampshire,U27,Public Research-Oriented,R1,Northeast,medium,81.82
1,2,3,Portland State University,U16,Public Research-Oriented,R2,West,large,80.30
2,3,3,Stanford University,U42,Private Research-Oriented,R1,West,medium,80.30
3,4,5,University of Texas at Austin,U2,Public Research-Oriented,R1,South,large,77.27
4,5,6,University of Notre Dame,U49,Private Research-Oriented,R1,Midwest,medium,75.76
...,...,...,...,...,...,...,...,...,...
74,75,95,Grinnell College,U72,Teaching-Oriented/Liberal Arts,-,Midwest,small,34.85
75,76,96,Jackson State University,U9,Public Research-Oriented,R2,South,small,31.82
76,77,97,Clark Atlanta University,U40,Private Research-Oriented,R2,South,small,28.79
77,78,99,Creighton University,U55,Private Research-Oriented,R2,Midwest,medium,27.27


,Institution,Count,Count.1,Faculty
0,1.0,► Carnegie Mellon University,34.3,92.0
1,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,2.0,► Univ. of Illinois at Urbana-Champaign,27.3,62.0
4,NaN,NaN,NaN,NaN
...,...,...,...,...
530,NaN,NaN,NaN,NaN
531,169.0,► University of Rhode Island,1.0,1.0
532,NaN,NaN,NaN,NaN
533,NaN,NaN,NaN,NaN


Wrote data/ACAI_CSRankings.csv
Wrote data/ACAI_CSRankings_unmatched.csv
Unmatched count, review manually: 49
